In [1]:
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

SQL Connection

In [2]:
load_dotenv("../.env")

db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

In [3]:
url = (
    f"postgresql+psycopg2://{db_user}:{db_password}"+
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(url)

Drawing tables from postgresql

In [4]:
tables = ["customers", "orders", "order_items", "order_payments", "order_reviews", "products", "sellers", "geolocation"]
data = dict()

for table in tables:
    data[table] = pd.read_sql(f"SELECT * FROM {table}", engine)

In [5]:
for table in tables:
    print(data[table].shape)

(99441, 5)
(99441, 8)
(112650, 7)
(103886, 5)
(99224, 7)
(32951, 9)
(3095, 4)
(1000163, 5)


In [6]:
df = pd.merge(data["orders"], data["customers"], on="customer_id", how="left")

In [7]:
df = pd.merge(df, data["order_items"], on="order_id", how="left")

In [8]:
df["order_item_id"].isnull().sum()

np.int64(775)

In [9]:
df[df["order_item_id"].isnull()]["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [10]:
df["order_status"].value_counts()

order_status
delivered      110197
shipped          1186
canceled          706
unavailable       610
invoiced          361
processing        357
created             5
approved            3
Name: count, dtype: int64

The null order_id values in the order_items table. This showings gives some idea about the cause of it.

In [11]:
df_revenue = df.dropna(subset=["order_item_id"])

In [12]:
df_revenue = pd.merge(df_revenue, data["order_payments"], on="order_id", how="left")

In [13]:
df_revenue["payment_type"].isnull().sum()

np.int64(3)

In [14]:
df_revenue[df_revenue["payment_type"].isnull()][["order_id","order_item_id","order_status","payment_type","price"]]

,order_id,order_item_id,order_status,payment_type,price
36410,bfbd0f9bdef84302105ad712db648a6c,1.0,delivered,NaN,44.99
36411,bfbd0f9bdef84302105ad712db648a6c,2.0,delivered,NaN,44.99
36412,bfbd0f9bdef84302105ad712db648a6c,3.0,delivered,NaN,44.99


This table shows that these 3 empty payment_type values are all coming from the same order.

In [15]:
df_revenue = df_revenue.dropna(subset=["payment_type"])

In [16]:
df_revenue = pd.merge(df_revenue, data["products"], on="product_id", how="left")

In [23]:
df_revenue = pd.merge(df_revenue, data["sellers"], on="seller_id", how="left")

Before merging order_reviews table, we check for the repeating order ids. Some orders might have more than one review

In [25]:
data["order_reviews"]["order_id"].duplicated().sum()

np.int64(551)

In [26]:
dup_order_id = data["order_reviews"][data["order_reviews"]["order_id"].duplicated(keep=False)]["order_id"].iloc[0]
data["order_reviews"][data["order_reviews"]["order_id"] == dup_order_id]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
30,540e7bbb2d06cfb7f85f3a88ba7ac97f,cf73e2cb1f4a9480ed70c154da3d954a,5,None,None,2018-01-18,2018-01-18 19:12:30
3115,aa193e76d35950c4ae988237bb36ed2b,cf73e2cb1f4a9480ed70c154da3d954a,5,None,None,2018-01-18,2018-01-18 17:36:45


We can see that there are different reviews for the same order. Since it would cause miscalculations after the merge, we'll keep just the last review by date

In [34]:
df_reviews_nodup = data["order_reviews"].sort_values("review_answer_timestamp").drop_duplicates(subset='order_id', keep='last')

In [35]:
df_reviews_nodup.shape

(98673, 7)

In [37]:
df_reviews_nodup[df_reviews_nodup["order_id"] == "cf73e2cb1f4a9480ed70c154da3d954a"]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
30,540e7bbb2d06cfb7f85f3a88ba7ac97f,cf73e2cb1f4a9480ed70c154da3d954a,5,None,None,2018-01-18,2018-01-18 19:12:30


Now we can merge it to the main table

In [43]:
df_revenue = pd.merge(df_revenue, df_reviews_nodup, on='order_id', how='left')

In [45]:
df_revenue["review_score"].isnull().sum()

np.int64(978)

In [49]:
df_revenue.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117601 entries, 0 to 117600
Data columns (total 39 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       117601 non-null  object        
 1   customer_id                    117601 non-null  object        
 2   order_status                   117601 non-null  object        
 3   order_purchase_timestamp       117601 non-null  datetime64[ns]
 4   order_approved_at              117586 non-null  datetime64[ns]
 5   order_delivered_carrier_date   116356 non-null  datetime64[ns]
 6   order_delivered_customer_date  115034 non-null  datetime64[ns]
 7   order_estimated_delivery_date  117601 non-null  datetime64[ns]
 8   customer_unique_id             117601 non-null  object        
 9   customer_zip_code_prefix       117601 non-null  int64         
 10  customer_city                  117601 non-null  object        
 11  

In [52]:
df_revenue.to_parquet("../data/processed/olist-processed-dataset.parquet", index=False)